# Manual 3-instrument XSPEC fitting notebook (XGA-style)

This notebook shows a manual XSPEC command-line workflow inspired by XGA.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import subprocess
import shutil
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass
class InstrumentSpectrum:
    name: str
    pha: Path
    bkg: Path
    rmf: Path
    arf: Path


@dataclass
class ThreeInstrumentSetup:
    mos1: InstrumentSpectrum
    mos2: InstrumentSpectrum
    pn: InstrumentSpectrum

    @property
    def all_specs(self):
        return [self.mos1, self.mos2, self.pn]


setup = ThreeInstrumentSetup(
    mos1=InstrumentSpectrum("mos1", Path("/path/to/m1_spec.pha"), Path("/path/to/m1_bkg.pha"), Path("/path/to/m1.rmf"), Path("/path/to/m1.arf")),
    mos2=InstrumentSpectrum("mos2", Path("/path/to/m2_spec.pha"), Path("/path/to/m2_bkg.pha"), Path("/path/to/m2.rmf"), Path("/path/to/m2.arf")),
    pn=InstrumentSpectrum("pn", Path("/path/to/pn_spec.pha"), Path("/path/to/pn_bkg.pha"), Path("/path/to/pn.rmf"), Path("/path/to/pn.arf")),
)

redshift = 0.2
nH_1e22 = 0.03
out_prefix = Path("./fit_outputs/cluster_example")
out_prefix.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
def build_xspec_script(
    setup: ThreeInstrumentSetup,
    out_prefix: Path,
    redshift: float,
    nH_1e22: float,
    kT_init_keV: float = 5.0,
    abundance_init_solar: float = 0.3,
    norm_init: float = 1e-3,
    fit_stat_delta: float = 1.0,
    lo_ignore_keV: float = 0.3,
    hi_ignore_keV: float = 10.0,
    xga_extract_tcl: Path = Path("xga/xspec_scripts/xga_extract.tcl"),
) -> Path:
    script_path = out_prefix.with_suffix(".xcm")
    out_base = str(out_prefix)

    lines = [
        f"source {xga_extract_tcl.resolve()}",
        "autosave off",
        "query yes",
        "setplot energy",
        "statistic cstat",
        "abund angr",
    ]

    for i, spec in enumerate(setup.all_specs, start=1):
        lines += [
            f'data {i}:1 "{spec.pha}"',
            f'backgrnd {i} "{spec.bkg}"',
            f'response {i} "{spec.rmf}"',
            f'arf {i} "{spec.arf}"',
            f"ignore {i}:**-{lo_ignore_keV} {hi_ignore_keV}-**",
        ]

    lines += [
        "model constant*tbabs*apec",
        "/*",
        "newpar 1 1.0",
        "freeze 1",
        f"newpar 2 {nH_1e22}",
        "freeze 2",
        f"newpar 3 {kT_init_keV}",
        f"newpar 4 {abundance_init_solar}",
        f"newpar 5 {redshift}",
        "freeze 5",
        f"newpar 6 {norm_init}",
        "newpar 7 1.0",
        "thaw 7",
        "newpar 8 1.0",
        "thaw 8",
        "fit 200",
        f"error {fit_stat_delta} 3 4 6",
        "plot ldata del",
        f'xga_extract "{out_base}" {{}} {redshift} 90 "constant*tbabs*apec" {{2}}',
        "exit",
    ]

    script_path.write_text("\n".join(lines) + "\n")
    return script_path


In [ ]:
def run_xspec_script(script_path: Path) -> subprocess.CompletedProcess:
    if shutil.which("xspec") is None:
        raise RuntimeError("XSPEC executable not found in PATH. Load HEASOFT/XSPEC first.")

    cmd = ["xspec", "-", str(script_path)]
    proc = subprocess.run(cmd, capture_output=True, text=True)

    log_path = script_path.with_suffix(".log")
    log_path.write_text(proc.stdout + "\n\nSTDERR:\n" + proc.stderr)

    if proc.returncode != 0:
        raise RuntimeError(f"XSPEC failed (return code={proc.returncode}). See log: {log_path}")
    return proc


In [ ]:
def extract_best_fit_parameters(results_csv: Path) -> dict:
    df = pd.read_csv(results_csv)
    if df.empty:
        raise ValueError(f"No rows in results file: {results_csv}")

    row = df.iloc[0].to_dict()
    keys = {k.lower(): k for k in row.keys()}

    def find_key(candidates):
        for cand in candidates:
            for k_low, k_orig in keys.items():
                if cand in k_low:
                    return k_orig
        return None

    kT_key = find_key(["kt", "apec_kt", "temperature"])
    ab_key = find_key(["abund", "apec_abund", "metal"])
    norm_key = find_key(["norm", "apec_norm"])

    return {
        "temperature_key": kT_key,
        "temperature_value": row.get(kT_key),
        "abundance_key": ab_key,
        "abundance_value": row.get(ab_key),
        "norm_key": norm_key,
        "norm_value": row.get(norm_key),
        "all_columns": list(df.columns),
    }


In [ ]:
def plot_spectrum_with_model(plot_csv: Path, title: str | None = None):
    df = pd.read_csv(plot_csv)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(df["X"], df["Y"], xerr=df["XERR"], yerr=df["YERR"], fmt="o", ms=3, alpha=0.7, label="Data")
    ax.plot(df["X"], df["YMODEL"], lw=2, label="Best-fit model")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Energy (keV)")
    ax.set_ylabel("Count rate")
    ax.legend()
    if title:
        ax.set_title(title)
    plt.show()


In [ ]:
script_path = build_xspec_script(setup, out_prefix, redshift, nH_1e22)
print(f"XSPEC script written to: {script_path}")

# run_xspec_script(script_path)
# results = extract_best_fit_parameters(out_prefix.with_name(out_prefix.name + "_results.csv"))
# results
# plot_spectrum_with_model(out_prefix.with_name(out_prefix.name + "_spec1.csv"), "MOS1")
# plot_spectrum_with_model(out_prefix.with_name(out_prefix.name + "_spec2.csv"), "MOS2")
# plot_spectrum_with_model(out_prefix.with_name(out_prefix.name + "_spec3.csv"), "PN")


## Notes
- `nH_1e22` is in units of `10^22 cm^-2` for `tbabs`.
- Redshift is frozen in this example.
- Output follows XGA extractor naming: `{prefix}_results.csv` and `{prefix}_specN.csv`.
